# Colab face-module test — `Rishabh060105/face_recog_complete_pipeline`

Run the cells in order and upload any `.mp4`, `.avi`, `.mov`, or `.mkv` video. Colab cannot open the script's `--show` window, so this notebook displays the annotated MP4 inline instead. An empty face database is valid, but every identity will be reported as `unknown`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path('/content/AI---Driven-Anamoly-Detection-System')
if not PROJECT.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'Rishabh060105/face_recog_complete_pipeline',
        'https://github.com/Retesh07/AI---Driven-Anamoly-Detection-System',
        str(PROJECT),
    ], check=True)
os.chdir(PROJECT)
print(PROJECT)

In [ ]:
%pip install -q ultralytics supervision opencv-contrib-python-headless

In [ ]:
import cv2

assert hasattr(cv2, 'FaceDetectorYN_create'), 'Install opencv-contrib-python-headless and rerun this cell.'
assert hasattr(cv2, 'FaceRecognizerSF_create'), 'Install opencv-contrib-python-headless and rerun this cell.'
print('OpenCV:', cv2.__version__, '| YuNet + SFace: available')

In [ ]:
os.chdir(PROJECT / 'threat_system')
subprocess.run([sys.executable, 'test_identity_stability.py'], check=True)
os.chdir(PROJECT)

In [ ]:
os.chdir(PROJECT / 'threat_system')
subprocess.run([sys.executable, 'download_face_models.py'], check=True)
os.chdir(PROJECT)

In [ ]:
from google.colab import files

os.chdir('/content')
uploaded = files.upload()
video_name = next((name for name in uploaded if Path(name).suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}), None)
if video_name is None:
    raise ValueError('Upload a video file with extension .mp4, .avi, .mov, or .mkv')
VIDEO = Path('/content') / video_name
print('Video:', VIDEO)

In [ ]:
FACE_DB = PROJECT / 'threat_system' / 'models' / 'faces'
OUTPUT = PROJECT / 'threat_system' / 'results' / 'face_eval'
FACE_DB.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT / 'threat_system')
subprocess.run([
    sys.executable, 'evaluate_face_module.py',
    '--video', str(VIDEO),
    '--face-db', str(FACE_DB),
    '--output', str(OUTPUT),
], check=True)

In [ ]:
import json
from IPython.display import Video, display

annotated = max(OUTPUT.glob('face_eval_*.mp4'), key=lambda path: path.stat().st_mtime)
preview = OUTPUT / 'preview_h264.mp4'
subprocess.run([
    'ffmpeg', '-y', '-i', str(annotated),
    '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
    str(preview),
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
summary_path = max(OUTPUT.glob('face_eval_*.json'), key=lambda path: path.stat().st_mtime)
print(json.loads(summary_path.read_text()))
display(Video(str(preview), embed=True, width=800))

## Full pipeline

`threat_system/main.py` also expects these model assets under `threat_system/models/`: `best_model_v3.pth`, `pose_features_v3/mean.npy`, `pose_features_v3/std.npy`, and `weapon_detector.pt`. They are not in this branch, so upload them before running the full command. Then run `python main.py --video <video> --output results -v` from `threat_system/`.